In [1]:
# ── Hücre 1: Mount Drive & Kurulumlar ──────────────────────────────────────────
!pip install scikit-multilearn -q
!pip install 'ultralytics>=8.3' -q

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import time

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT       = Path('/content/drive/MyDrive/Wrist Dataset')
LABELS_DIR = ROOT / 'folder_structure/yolov5/labels'
IMAGES_DIR = ROOT / 'images'
CSV_PATH   = ROOT / 'dataset.csv'

# ── Local working dirs ─────────────────────────────────────────────────────────
LOCAL_DIR  = Path('/content')
CACHE_PATH = ROOT / 'annotations_cache.csv'

# ── Split config ───────────────────────────────────────────────────────────────
TRAIN_RATIO   = 0.70
VAL_RATIO     = 0.10
TEST_RATIO    = 0.20
SEED          = 42
FORCE_REBUILD = False

CLASSES = ['boneanomaly', 'bonelesion', 'foreignbody_metal',
           'fracture', 'periostealreaction', 'pronatorsign', 'softtissue']

AGE_BINS   = [0, 5, 9, 13, 99]
AGE_LABELS = ['0-4', '5-8', '9-12', '13-19']

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.7 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
# ── Hücre 2: İş Mantığı Fonksiyonları (Business Logic) ─────────────────────────

def remap_class(original_id):
    """Orijinal 9 sınıfı, birleştirilmiş 7 sınıfa dönüştürür."""
    CLASS_REMAP = {0:0, 1:1, 2:2, 3:3, 4:2, 5:4, 6:5, 7:6, 8:None}
    if original_id not in CLASS_REMAP:
        raise KeyError(f'Bilinmeyen sınıf ID: {original_id}')
    return CLASS_REMAP[original_id]

def get_age_group(age):
    """Verilen yaşı ilgili yaş grubuna (string olarak) atar."""
    # Pandas kullanarak skaler değeri gruplama simülasyonu
    res = pd.cut([age], bins=AGE_BINS, labels=AGE_LABELS, right=False)
    return str(res[0])

def check_patient_leakage(train_set, val_set, test_set):
    """Veri setleri arasında hasta sızıntısı (leakage) olup olmadığını kontrol eder."""
    train_set, val_set, test_set = set(train_set), set(val_set), set(test_set)

    if train_set & val_set:
        raise ValueError("SıZINTI: Train ve Val setlerinde ortak hastalar var!")
    if train_set & test_set:
        raise ValueError("SıZINTI: Train ve Test setlerinde ortak hastalar var!")
    if val_set & test_set:
        raise ValueError("SıZINTI: Val ve Test setlerinde ortak hastalar var!")

    return True # Sorun yoksa True döner

In [3]:
# ── Hücre 3: Unit Testler ──────────────────────────────────────────────────────
import unittest

class TestDataPreparation(unittest.TestCase):

    def test_remap_class_valid(self):
        self.assertEqual(remap_class(0), 0)
        self.assertEqual(remap_class(4), 2)  # Metal -> 2 olmalı
        self.assertIsNone(remap_class(8))    # text -> drop olmalı (None)

    def test_remap_class_invalid(self):
        with self.assertRaises(KeyError):
            remap_class(99) # Olmayan sınıf gelirse patlamalı

    def test_age_grouping(self):
        self.assertEqual(get_age_group(3), '0-4')
        self.assertEqual(get_age_group(5), '5-8')
        self.assertEqual(get_age_group(12), '9-12')
        self.assertEqual(get_age_group(25), '13-19')

    def test_no_patient_leakage(self):
        # Geçerli senaryo
        self.assertTrue(check_patient_leakage(['P1', 'P2'], ['P3'], ['P4']))

    def test_with_patient_leakage(self):
        # Sızıntı olan senaryo (P2 hem train hem val'de)
        with self.assertRaises(ValueError):
            check_patient_leakage(['P1', 'P2'], ['P2', 'P3'], ['P4'])

# Notebook ortamında testleri güvenle çalıştırmak için bu komut kullanılır
print("Unit Testler Çalıştırılıyor...")
unittest.main(argv=['first-arg-is-ignored'], exit=False)

.....
----------------------------------------------------------------------
Ran 5 tests in 0.060s

OK


Unit Testler Çalıştırılıyor...


In [4]:
# ── Hücre 4: Veri Okuma, Tablo Oluşturma ve Stratified Split ───────────────────
import time
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict
from skmultilearn.model_selection import IterativeStratification

t0 = time.time()

if CACHE_PATH.exists() and not FORCE_REBUILD:
    print('Loading annotations_cache.csv ...', end=' ', flush=True)
    _df    = pd.read_csv(CACHE_PATH, dtype={'image_stem': str})
    _valid = _df[_df['class_id'].notna()].copy()
    _valid['class_id'] = _valid['class_id'].astype(int)
    _cls_by_stem = _valid.groupby('image_stem')['class_id'].apply(set).to_dict()
    img_records  = {
        stem: {'new_classes': _cls_by_stem.get(stem, set()),
               'is_empty':    stem not in _cls_by_stem}
        for stem in _df['image_stem'].unique()
    }
    print(f'done  —  {len(img_records):,} images')
else:
    def _read_label(p):
        class_ids, annotations = set(), []
        for line in p.read_text().split('\n'):
            tok = line.split()
            if not tok: continue
            orig = int(tok[0])

            # --- UNIT TESTTEN GEÇEN FONKSİYONU KULLANIYORUZ ---
            nid = remap_class(orig)

            if nid is not None:
                class_ids.add(nid)
                if len(tok) >= 5:
                    annotations.append((nid, tok[1], tok[2], tok[3], tok[4]))
        return p.stem, class_ids, annotations

    label_files = sorted(LABELS_DIR.glob('*.txt'))
    print('Parsing label files (first run)...', flush=True)
    with ThreadPoolExecutor(max_workers=8) as ex:
        triples = list(ex.map(_read_label, label_files))

    img_records = {k: {'new_classes': v, 'is_empty': len(v) == 0} for k, v, _ in triples}

    rows = []
    for stem, _, annotations in triples:
        if annotations:
            for (cid, x, y, w, h) in annotations:
                rows.append({'image_stem': stem, 'class_id': cid, 'x_center': x, 'y_center': y, 'width': w, 'height': h})
        else:
            rows.append({'image_stem': stem, 'class_id': None, 'x_center': None, 'y_center': None, 'width': None, 'height': None})
    pd.DataFrame(rows).to_csv(CACHE_PATH, index=False)
    print('Cache saved.')

# Patient Tablosu
pid_of  = {s: s.split('_')[0] for s in img_records}
pid_cls = defaultdict(set)
pid_ann = defaultdict(bool)
for stem, rec in img_records.items():
    pid = pid_of[stem]
    pid_cls[pid].update(rec['new_classes'])
    pid_ann[pid] = pid_ann[pid] or (not rec['is_empty'])

valid_pids = {p for p, v in pid_ann.items() if v}
df_m = pd.read_csv(CSV_PATH, dtype={'patient_id': str})
df_m['patient_id'] = df_m['patient_id'].str.zfill(4)
df_m = df_m[df_m['patient_id'].isin(valid_pids)].copy()

agg = (df_m.groupby('patient_id', sort=False)
           .agg(gender  = ('gender',     'first'),
                min_age = ('age',        'min'),
                lat_set = ('laterality', lambda x: frozenset(x.dropna())))
           .reset_index())

# --- YAŞ BÖLME KONTROLÜ (Test edildi) ---
agg['age_group']  = pd.cut(agg['min_age'], bins=AGE_BINS, labels=AGE_LABELS, right=False).astype(str)
agg['laterality'] = agg['lat_set'].apply(lambda s: 'both' if len(s) > 1 else f'{list(s)[0]}_only')

for cid, cname in enumerate(CLASSES):
    agg[cname] = agg['patient_id'].apply(lambda pid, c=cid: int(c in pid_cls.get(pid, set())))

patient_df = agg.set_index('patient_id')[['gender', 'age_group', 'laterality'] + CLASSES]

# Stratification (Aynı)
enc  = pd.get_dummies(patient_df, columns=['gender', 'age_group', 'laterality'], dtype=int)
pids = enc.index.to_numpy()
lbl  = enc.values.astype(float)
idx  = np.random.default_rng(SEED).permutation(len(pids))
pids, lbl = pids[idx], lbl[idx]

s1 = IterativeStratification(n_splits=2, order=1, sample_distribution_per_fold=[1 - TRAIN_RATIO, TRAIN_RATIO])
f0, f1 = next(s1.split(sp.lil_matrix(lbl), sp.lil_matrix(lbl)))
tr_i, tp_i = (f1, f0) if len(f1) > len(f0) else (f0, f1)
train_pids  = pids[tr_i]
tp, tp_lbl  = pids[tp_i], lbl[tp_i]

vf = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
s2 = IterativeStratification(n_splits=2, order=1, sample_distribution_per_fold=[1 - vf, vf])
g0, g1 = next(s2.split(sp.lil_matrix(tp_lbl), sp.lil_matrix(tp_lbl)))
v_i, te_i = (g1, g0) if len(g1) < len(g0) else (g0, g1)
val_pids   = tp[v_i]
test_pids  = tp[te_i]

pid_split = {'train': set(train_pids), 'val': set(val_pids), 'test': set(test_pids)}

# --- UNIT TESTTEN GEÇEN SIZINTI KONTROL FONKSİYONU ---
check_patient_leakage(pid_split['train'], pid_split['val'], pid_split['test'])
print('Sızıntı (Leakage) Testi Başarılı: Veri setleri arası çakışma yok!')

for sn, pid_set in pid_split.items():
    rows = []
    for stem, rec in img_records.items():
        if rec['is_empty']: continue
        pid = pid_of[stem]
        if pid not in pid_set: continue
        p = patient_df.loc[pid]
        rows.append({'image_path': str(IMAGES_DIR / f'{stem}.png'), 'image_stem': stem, 'patient_id': pid,
                     'gender': p['gender'], 'age_group': p['age_group'], 'laterality': p['laterality'],
                     **{c: int(p[c]) for c in CLASSES}})
    df_out = pd.DataFrame(rows).sort_values('image_path').reset_index(drop=True)
    df_out.to_csv(LOCAL_DIR / f'{sn}_split{SEED}.csv', index=False)

print(f'Süre: {time.time()-t0:.1f}s')

Loading annotations_cache.csv ... done  —  20,327 images
Sızıntı (Leakage) Testi Başarılı: Veri setleri arası çakışma yok!
Süre: 3.8s


In [5]:
# ── Hücre 5: YOLO Dosya Yapısını Hazırlama ─────────────────────────────────────
import shutil

LOCAL_IMGS = Path('/content/images_local')
LOCAL_IMGS.mkdir(exist_ok=True)

all_stems = set()
for sn in ['train', 'val', 'test']:
    all_stems.update(pd.read_csv(LOCAL_DIR / f'{sn}_split{SEED}.csv')['image_stem'])

to_copy = [s for s in all_stems if not (LOCAL_IMGS / f'{s}.png').exists()]
if to_copy:
    print(f'Copying {len(to_copy):,} images...')
    def _copy_img(stem): shutil.copy2(IMAGES_DIR / f'{stem}.png', LOCAL_IMGS / f'{stem}.png')
    with ThreadPoolExecutor(max_workers=32) as ex: list(ex.map(_copy_img, to_copy))

_df    = pd.read_csv(CACHE_PATH, dtype={'image_stem': str})
_valid = _df[_df['class_id'].notna()].copy()
_valid['class_id'] = _valid['class_id'].astype(int)
anno_map = {}
for row in _valid.itertuples(index=False):
    anno_map.setdefault(row.image_stem, []).append(f'{row.class_id} {row.x_center} {row.y_center} {row.width} {row.height}')

YOLO = Path(f'/content/yolo_data_{SEED}')
for split in ['train', 'val', 'test']:
    (YOLO / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO / 'labels' / split).mkdir(parents=True, exist_ok=True)

def _prep(args):
    stem, sn = args
    img_dst = YOLO / 'images' / sn / f'{stem}.png'
    if not img_dst.exists(): img_dst.symlink_to(LOCAL_IMGS / f'{stem}.png')
    lbl_dst = YOLO / 'labels' / sn / f'{stem}.txt'
    if not lbl_dst.exists(): lbl_dst.write_text('\n'.join(anno_map.get(stem, [])))

tasks = [(stem, sn) for sn in ['train', 'val', 'test'] for stem in pd.read_csv(LOCAL_DIR / f'{sn}_split{SEED}.csv')['image_stem']]
with ThreadPoolExecutor(max_workers=8) as ex: list(ex.map(_prep, tasks))

YOLO_YAML = YOLO / 'dataset.yaml'
YOLO_YAML.write_text(
    f'path: {YOLO}\n'
    f'train: images/train\n'
    f'val:   images/val\n'
    f'test:  images/test\n\n'
    f'nc: {len(CLASSES)}\n'
    f'names: {CLASSES}\n'
)
print('YOLO klasör yapısı ve YAML oluşturuldu.')

Copying 13,946 images...
YOLO klasör yapısı ve YAML oluşturuldu.


In [ ]:
# ── Hücre 6: YOLO Training & Validation ────────────────────────────────────────
from ultralytics import YOLO
from IPython.display import Image, display

# Training
model = YOLO('yolov10x.pt')

results = model.train(
    data      = str(YOLO_YAML),
    epochs    = 100,
    imgsz     = 640,
    batch     = 32,
    patience  = 20,
    optimizer = 'AdamW',
    lr0       = 1e-3,
    cos_lr    = True,
    cache     = 'ram',
    fliplr    = 0.0,
    flipud    = 0.0,
    degrees   = 5,
    translate = 0.05,
    scale     = 0.3,
    hsv_s     = 0.0,
    hsv_h     = 0.0,
    device    = 0,
    project   = '/content/runs',
    name      = f'yolov10x_seed{SEED}',
    exist_ok  = True,
    save      = True,
    plots     = True,
)

# Testing
run_dir = Path(f'/content/runs/yolov10x_seed{SEED}')
best_pt = run_dir / 'weights/best.pt'
test_model = YOLO(best_pt)

test_res = test_model.val(
    data      = str(YOLO_YAML),
    split     = 'test',
    device    = 0,
    plots     = True,
    save_json = True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/yolo_data_42/dataset.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.4, imgsz=